# MM60 — Centro 4128 — Datalake

**Tabela:** `dev_procurement.corp_curated.tbl_ds_mdm_mm60`
**Domínio:** Mestre de materiais por centro
**Filtro do cenário:** `cod_centro = '4128'`
**Colunas:** 17 · **Clustering declarado:** `cod_material`, `cod_centro`

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente com o filtro do centro embutido. Não há widget, view temporária nem ordem obrigatória.

## Objetivo

Extrair e caracterizar **toda** a base do centro 4128 para comparação com o extrato do SAP.

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados da tabela |
| 2 | Volumetria e representatividade do cenário |
| 3 | Confirmação do filtro |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| 6 | Preenchimento de todas as colunas |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 9 | Perfil numérico |
| **10** | **Totais para conciliação com o SAP** |
| 11 | Datas |
| 12 | Códigos e zeros à esquerda |
| **13** | **Chaves normalizadas para join** |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| 18 | Análises específicas |
| **19** | **EXTRAÇÃO COMPLETA** |
| 20 | Resumo do cenário |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados da tabela

In [ ]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_mdm_mm60;

In [ ]:
-- Formato, tamanho e particoes (falha se nao for Delta)
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_mdm_mm60;

In [ ]:
-- Ultimas gravacoes (falha se for view)
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_mdm_mm60 LIMIT 20;

## 2. Volumetria e representatividade

Quanto o centro 4128 representa do total da tabela.

In [ ]:
-- 2. VOLUMETRIA DO CENARIO
SELECT
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60)                                   AS linhas_tabela_toda,
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128')                               AS linhas_centro_4128,
  ROUND(100.0 * (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128')
              / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60), 4)                  AS pct_do_total,
  (SELECT COUNT(DISTINCT `cod_centro`) FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60)                     AS centros_na_tabela;

## 3. Confirmação do filtro

Confirma que o valor `4128` existe e que não há variação de formato
(espaços, zeros à esquerda) que faça o filtro perder linhas silenciosamente.

**Se retornar mais de uma linha, o filtro `= '4128'` está incompleto.**

In [ ]:
-- 3. O FILTRO PEGOU TUDO?
SELECT CAST(`cod_centro` AS STRING)                    AS valor_bruto,
       length(CAST(`cod_centro` AS STRING))            AS comprimento,
       COUNT(*)                                   AS linhas
FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60
WHERE regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '') = '4128'
   OR trim(CAST(`cod_centro` AS STRING)) = '4128'
GROUP BY CAST(`cod_centro` AS STRING), length(CAST(`cod_centro` AS STRING))
ORDER BY linhas DESC;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional
multiplicando as linhas.

In [ ]:
-- 4. GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'),
g AS (
  SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'cod_material + cod_centro + tp_avaliacao' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `tp_avaliacao` FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'cod_material' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material` FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128')
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `cod_material + cod_centro`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [ ]:
-- 5. CHAVES DUPLICADAS
SELECT `cod_material`, `cod_centro`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
GROUP BY `cod_material`, `cod_centro`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [ ]:
-- 5.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH cen AS (
  SELECT * FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
),
dup AS (
  SELECT `cod_material`, `cod_centro` FROM cen GROUP BY `cod_material`, `cod_centro` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM cen c JOIN dup ON c.`cod_material` <=> dup.`cod_material` AND c.`cod_centro` <=> dup.`cod_centro`
),
agg AS (
  SELECT `cod_material`, `cod_centro`,
         COUNT(DISTINCT `tp_avaliacao`) AS `tp_avaliacao`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `sg_um_basica`) AS `sg_um_basica`,
         COUNT(DISTINCT `tp_material`) AS `tp_material`,
         COUNT(DISTINCT `cod_grupo_comprador`) AS `cod_grupo_comprador`,
         COUNT(DISTINCT `cod_grupo_mercadoria`) AS `cod_grupo_mercadoria`,
         COUNT(DISTINCT `nm_criado_por`) AS `nm_criado_por`,
         COUNT(DISTINCT `vl_preco_brl`) AS `vl_preco_brl`,
         COUNT(DISTINCT `sg_moeda`) AS `sg_moeda`,
         COUNT(DISTINCT `dt_ultima_modificacao`) AS `dt_ultima_modificacao`,
         COUNT(DISTINCT `tp_mrp`) AS `tp_mrp`,
         COUNT(DISTINCT `cod_abc`) AS `cod_abc`,
         COUNT(DISTINCT `cod_classe_avaliacao`) AS `cod_classe_avaliacao`,
         COUNT(DISTINCT `tp_controle_preco`) AS `tp_controle_preco`,
         COUNT(DISTINCT `qt_unidade_preco`) AS `qt_unidade_preco`
  FROM d GROUP BY `cod_material`, `cod_centro`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(15,
    'tp_avaliacao', MAX(`tp_avaliacao`),
    'desc_material', MAX(`desc_material`),
    'sg_um_basica', MAX(`sg_um_basica`),
    'tp_material', MAX(`tp_material`),
    'cod_grupo_comprador', MAX(`cod_grupo_comprador`),
    'cod_grupo_mercadoria', MAX(`cod_grupo_mercadoria`),
    'nm_criado_por', MAX(`nm_criado_por`),
    'vl_preco_brl', MAX(`vl_preco_brl`),
    'sg_moeda', MAX(`sg_moeda`),
    'dt_ultima_modificacao', MAX(`dt_ultima_modificacao`),
    'tp_mrp', MAX(`tp_mrp`),
    'cod_abc', MAX(`cod_abc`),
    'cod_classe_avaliacao', MAX(`cod_classe_avaliacao`),
    'tp_controle_preco', MAX(`tp_controle_preco`),
    'qt_unidade_preco', MAX(`qt_unidade_preco`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada **neste centro**.

Uma coluna pode ter dado na tabela toda e estar vazia no centro 4128 — ou o contrário.
Por isso a varredura é feita sobre o recorte, não sobre a base completa.

In [ ]:
-- 6. PREENCHIMENTO NO CENTRO 4128
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'),
perf AS (
  SELECT stack(17,
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'tp_avaliacao', 'string', COUNT_IF(`tp_avaliacao` IS NULL), COUNT_IF(`tp_avaliacao` IS NOT NULL AND lower(trim(`tp_avaliacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_avaliacao`) RLIKE '^0+([.,]0+)?$'),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'sg_um_basica', 'string', COUNT_IF(`sg_um_basica` IS NULL), COUNT_IF(`sg_um_basica` IS NOT NULL AND lower(trim(`sg_um_basica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_um_basica`) RLIKE '^0+([.,]0+)?$'),
    'tp_material', 'string', COUNT_IF(`tp_material` IS NULL), COUNT_IF(`tp_material` IS NOT NULL AND lower(trim(`tp_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_comprador', 'string', COUNT_IF(`cod_grupo_comprador` IS NULL), COUNT_IF(`cod_grupo_comprador` IS NOT NULL AND lower(trim(`cod_grupo_comprador`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_comprador`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_mercadoria', 'string', COUNT_IF(`cod_grupo_mercadoria` IS NULL), COUNT_IF(`cod_grupo_mercadoria` IS NOT NULL AND lower(trim(`cod_grupo_mercadoria`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_mercadoria`) RLIKE '^0+([.,]0+)?$'),
    'nm_criado_por', 'string', COUNT_IF(`nm_criado_por` IS NULL), COUNT_IF(`nm_criado_por` IS NOT NULL AND lower(trim(`nm_criado_por`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_criado_por`) RLIKE '^0+([.,]0+)?$'),
    'vl_preco_brl', 'decimal(18,2)', COUNT_IF(`vl_preco_brl` IS NULL), 0L, COUNT_IF(`vl_preco_brl` = 0),
    'sg_moeda', 'string', COUNT_IF(`sg_moeda` IS NULL), COUNT_IF(`sg_moeda` IS NOT NULL AND lower(trim(`sg_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_moeda`) RLIKE '^0+([.,]0+)?$'),
    'dt_ultima_modificacao', 'string', COUNT_IF(`dt_ultima_modificacao` IS NULL), COUNT_IF(`dt_ultima_modificacao` IS NOT NULL AND lower(trim(`dt_ultima_modificacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^0+([.,]0+)?$'),
    'tp_mrp', 'string', COUNT_IF(`tp_mrp` IS NULL), COUNT_IF(`tp_mrp` IS NOT NULL AND lower(trim(`tp_mrp`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_mrp`) RLIKE '^0+([.,]0+)?$'),
    'cod_abc', 'string', COUNT_IF(`cod_abc` IS NULL), COUNT_IF(`cod_abc` IS NOT NULL AND lower(trim(`cod_abc`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_abc`) RLIKE '^0+([.,]0+)?$'),
    'cod_classe_avaliacao', 'string', COUNT_IF(`cod_classe_avaliacao` IS NULL), COUNT_IF(`cod_classe_avaliacao` IS NOT NULL AND lower(trim(`cod_classe_avaliacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_classe_avaliacao`) RLIKE '^0+([.,]0+)?$'),
    'tp_controle_preco', 'string', COUNT_IF(`tp_controle_preco` IS NULL), COUNT_IF(`tp_controle_preco` IS NOT NULL AND lower(trim(`tp_controle_preco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_controle_preco`) RLIKE '^0+([.,]0+)?$'),
    'qt_unidade_preco', 'decimal(5,0)', COUNT_IF(`qt_unidade_preco` IS NULL), 0L, COUNT_IF(`qt_unidade_preco` = 0)
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros,
       t.total - p.nulos - p.vazios - p.zeros                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                       THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                              AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade no cenário

In [ ]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'),
card AS (
  SELECT stack(17,
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'tp_avaliacao', 'string', approx_count_distinct(`tp_avaliacao`),
    'desc_material', 'string', approx_count_distinct(`desc_material`),
    'sg_um_basica', 'string', approx_count_distinct(`sg_um_basica`),
    'tp_material', 'string', approx_count_distinct(`tp_material`),
    'cod_grupo_comprador', 'string', approx_count_distinct(`cod_grupo_comprador`),
    'cod_grupo_mercadoria', 'string', approx_count_distinct(`cod_grupo_mercadoria`),
    'nm_criado_por', 'string', approx_count_distinct(`nm_criado_por`),
    'vl_preco_brl', 'decimal(18,2)', approx_count_distinct(`vl_preco_brl`),
    'sg_moeda', 'string', approx_count_distinct(`sg_moeda`),
    'dt_ultima_modificacao', 'string', approx_count_distinct(`dt_ultima_modificacao`),
    'tp_mrp', 'string', approx_count_distinct(`tp_mrp`),
    'cod_abc', 'string', approx_count_distinct(`cod_abc`),
    'cod_classe_avaliacao', 'string', approx_count_distinct(`cod_classe_avaliacao`),
    'tp_controle_preco', 'string', approx_count_distinct(`tp_controle_preco`),
    'qt_unidade_preco', 'decimal(5,0)', approx_count_distinct(`qt_unidade_preco`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma, dentro do cenário.

In [ ]:
-- 8. DOMINIO DAS CATEGORICAS
(SELECT 'tp_avaliacao' AS coluna, CAST(`tp_avaliacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128' GROUP BY `tp_avaliacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_um_basica' AS coluna, CAST(`sg_um_basica` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128' GROUP BY `sg_um_basica` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_material' AS coluna, CAST(`tp_material` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128' GROUP BY `tp_material` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_comprador' AS coluna, CAST(`cod_grupo_comprador` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128' GROUP BY `cod_grupo_comprador` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_mercadoria' AS coluna, CAST(`cod_grupo_mercadoria` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128' GROUP BY `cod_grupo_mercadoria` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_moeda' AS coluna, CAST(`sg_moeda` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128' GROUP BY `sg_moeda` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_mrp' AS coluna, CAST(`tp_mrp` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128' GROUP BY `tp_mrp` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_abc' AS coluna, CAST(`cod_abc` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128' GROUP BY `cod_abc` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_classe_avaliacao' AS coluna, CAST(`cod_classe_avaliacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128' GROUP BY `cod_classe_avaliacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_controle_preco' AS coluna, CAST(`tp_controle_preco` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128' GROUP BY `tp_controle_preco` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

Campos `double` exigem tolerância de 0,005 na comparação com o SAP.

In [ ]:
-- 9. PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(2,
    'vl_preco_brl', 'decimal(18,2)', COUNT(`vl_preco_brl`), CAST(MIN(`vl_preco_brl`) AS DOUBLE), CAST(MAX(`vl_preco_brl`) AS DOUBLE), CAST(AVG(`vl_preco_brl`) AS DOUBLE), CAST(percentile_approx(`vl_preco_brl`, 0.5) AS DOUBLE), COUNT_IF(`vl_preco_brl` < 0), COUNT_IF(`vl_preco_brl` = 0),
    'qt_unidade_preco', 'decimal(5,0)', COUNT(`qt_unidade_preco`), CAST(MIN(`qt_unidade_preco`) AS DOUBLE), CAST(MAX(`qt_unidade_preco`) AS DOUBLE), CAST(AVG(`qt_unidade_preco`) AS DOUBLE), CAST(percentile_approx(`qt_unidade_preco`, 0.5) AS DOUBLE), COUNT_IF(`qt_unidade_preco` < 0), COUNT_IF(`qt_unidade_preco` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 10. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel extraído do SAP e compare linha a linha.
Divergência de total é o teste mais rápido para detectar registro faltando ou duplicado —
e cobre o ponto cego da contagem de linhas, que sozinha não prova nada.

In [ ]:
-- 10. TOTAIS PARA CONCILIACAO
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(2,
    'vl_preco_brl', CAST(SUM(`vl_preco_brl`) AS DOUBLE), CAST(ROUND(SUM(`vl_preco_brl`), 2) AS STRING), COUNT(`vl_preco_brl`),
    'qt_unidade_preco', CAST(SUM(`qt_unidade_preco`) AS DOUBLE), CAST(ROUND(SUM(`qt_unidade_preco`), 2) AS STRING), COUNT(`qt_unidade_preco`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 11. Datas armazenadas como STRING

**Armadilha:** o SAP exporta `2024-02-23 00:00:00` e o Datalake grava `20240223`.
Mesma data, formato diferente — normalizar para `AAAAMMDD` antes de comparar.

In [ ]:
-- 11. DATAS EM STRING
SELECT coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, minimo, maximo,
       CASE WHEN (CASE WHEN fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato' ELSE 'formato unico' END AS veredito
FROM (
  SELECT stack(1,
    'dt_ultima_modificacao', COUNT_IF(`dt_ultima_modificacao` IS NULL OR trim(`dt_ultima_modificacao`) = ''), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_ultima_modificacao`) NOT IN ('', '00000000') THEN `dt_ultima_modificacao` END), MAX(CASE WHEN trim(`dt_ultima_modificacao`) NOT IN ('', '00000000') THEN `dt_ultima_modificacao` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, minimo, maximo)
  FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 12. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

In [ ]:
-- 12. CODIGOS
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(3,
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'tp_avaliacao', 'string', COUNT_IF(CAST(`tp_avaliacao` AS STRING) IS NULL OR trim(CAST(`tp_avaliacao` AS STRING)) = ''), MIN(length(trim(CAST(`tp_avaliacao` AS STRING)))), MAX(length(trim(CAST(`tp_avaliacao` AS STRING)))), COUNT_IF(trim(CAST(`tp_avaliacao` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`tp_avaliacao` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`tp_avaliacao` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 13. Chaves normalizadas para join com o SAP

Lista das chaves já **sem zeros à esquerda**, prontas para colar no Excel
e cruzar com o extrato do SAP via PROCV/ÍNDICE.

Baixe como CSV e use para identificar registros presentes de um lado e ausentes do outro.

In [ ]:
-- 13. CHAVES NORMALIZADAS (para cruzar com o SAP)
SELECT DISTINCT
       regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS `cod_material_norm`,
       regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '') AS `cod_centro_norm`
FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
ORDER BY 1, 2;

## 14. Checksum de linha

Gera uma impressão digital de cada linha. Dois usos:

- **Contar linhas realmente distintas** — se `linhas` for maior que `linhas_unicas`,
  existem registros 100% idênticos (duplicata real)
- **Comparação rápida** — aplicando a mesma concatenação no SAP, dá para achar
  divergências sem comparar campo a campo

In [ ]:
-- 14. CHECKSUM DE LINHA
SELECT COUNT(*)                                                 AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`tp_avaliacao` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`sg_um_basica` AS STRING), ''), COALESCE(CAST(`tp_material` AS STRING), ''), COALESCE(CAST(`cod_grupo_comprador` AS STRING), ''), COALESCE(CAST(`cod_grupo_mercadoria` AS STRING), ''), COALESCE(CAST(`nm_criado_por` AS STRING), ''), COALESCE(CAST(`vl_preco_brl` AS STRING), ''), COALESCE(CAST(`sg_moeda` AS STRING), ''), COALESCE(CAST(`dt_ultima_modificacao` AS STRING), ''), COALESCE(CAST(`tp_mrp` AS STRING), ''), COALESCE(CAST(`cod_abc` AS STRING), ''), COALESCE(CAST(`cod_classe_avaliacao` AS STRING), ''), COALESCE(CAST(`tp_controle_preco` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''))))             AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`tp_avaliacao` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`sg_um_basica` AS STRING), ''), COALESCE(CAST(`tp_material` AS STRING), ''), COALESCE(CAST(`cod_grupo_comprador` AS STRING), ''), COALESCE(CAST(`cod_grupo_mercadoria` AS STRING), ''), COALESCE(CAST(`nm_criado_por` AS STRING), ''), COALESCE(CAST(`vl_preco_brl` AS STRING), ''), COALESCE(CAST(`sg_moeda` AS STRING), ''), COALESCE(CAST(`dt_ultima_modificacao` AS STRING), ''), COALESCE(CAST(`tp_mrp` AS STRING), ''), COALESCE(CAST(`cod_abc` AS STRING), ''), COALESCE(CAST(`cod_classe_avaliacao` AS STRING), ''), COALESCE(CAST(`tp_controle_preco` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''))))  AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`tp_avaliacao` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`sg_um_basica` AS STRING), ''), COALESCE(CAST(`tp_material` AS STRING), ''), COALESCE(CAST(`cod_grupo_comprador` AS STRING), ''), COALESCE(CAST(`cod_grupo_mercadoria` AS STRING), ''), COALESCE(CAST(`nm_criado_por` AS STRING), ''), COALESCE(CAST(`vl_preco_brl` AS STRING), ''), COALESCE(CAST(`sg_moeda` AS STRING), ''), COALESCE(CAST(`dt_ultima_modificacao` AS STRING), ''), COALESCE(CAST(`tp_mrp` AS STRING), ''), COALESCE(CAST(`cod_abc` AS STRING), ''), COALESCE(CAST(`cod_classe_avaliacao` AS STRING), ''), COALESCE(CAST(`tp_controle_preco` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128';

## 15. Amostra de linhas completas

In [ ]:
-- 15. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
ORDER BY `cod_material`
LIMIT 20;

## 16. Distribuição interna do centro 4128

Como o volume se reparte dentro do cenário. Útil para conferir se o extrato do SAP
tem a mesma composição.

In [ ]:
-- 16. DISTRIBUICAO POR tp_material
SELECT COALESCE(NULLIF(trim(CAST(`tp_material` AS STRING)), ''), '(vazio)') AS `tp_material`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_material` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR cod_grupo_mercadoria
SELECT COALESCE(NULLIF(trim(CAST(`cod_grupo_mercadoria` AS STRING)), ''), '(vazio)') AS `cod_grupo_mercadoria`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_grupo_mercadoria` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR tp_mrp
SELECT COALESCE(NULLIF(trim(CAST(`tp_mrp` AS STRING)), ''), '(vazio)') AS `tp_mrp`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_mrp` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR cod_abc
SELECT COALESCE(NULLIF(trim(CAST(`cod_abc` AS STRING)), ''), '(vazio)') AS `cod_abc`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_abc` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

In [ ]:
-- 17. FRESHNESS
-- Tabela sem coluna de data de ingestao. Use o historico de gravacao.
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_mdm_mm60 LIMIT 10;

## 18. Análises específicas — MM60

### 18.1 Split valuation no centro 4128

Se um material tem mais de um `tp_avaliacao`, a chave material+centro deixa de ser única.

In [ ]:
-- 18.1 SPLIT VALUATION
SELECT qt_avaliacoes, COUNT(*) AS materiais, SUM(linhas - 1) AS linhas_excedentes
FROM (
  SELECT cod_material, COUNT(DISTINCT tp_avaliacao) AS qt_avaliacoes, COUNT(*) AS linhas
  FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
  GROUP BY cod_material
)
GROUP BY qt_avaliacoes
ORDER BY qt_avaliacoes;

In [ ]:
-- 18.1b MATERIAIS COM SPLIT VALUATION
SELECT cod_material,
       COUNT(DISTINCT tp_avaliacao) AS qt_avaliacoes,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(tp_avaliacao))) AS tipos
FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
GROUP BY cod_material
HAVING COUNT(DISTINCT tp_avaliacao) > 1
ORDER BY qt_avaliacoes DESC
LIMIT 25;

### 18.2 Preço por tipo de controle

In [ ]:
-- 18.2 PRECO
SELECT tp_controle_preco,
       COUNT(*) AS materiais,
       COUNT_IF(vl_preco_brl IS NULL OR vl_preco_brl = 0) AS sem_preco,
       ROUND(AVG(vl_preco_brl), 2) AS preco_medio,
       ROUND(SUM(vl_preco_brl), 2) AS soma_precos
FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'
GROUP BY tp_controle_preco
ORDER BY materiais DESC;

## 19. EXTRAÇÃO COMPLETA — centro 4128

**Esta é a célula que você baixa para comparar com o SAP.**

Após executar, use **Download → CSV** no resultado.

> **Limites do Databricks:** a tela mostra até 10.000 linhas, mas o download em CSV
> vai além disso. Se o volume for muito grande, use a célula 19.1.

In [ ]:
-- 19. EXTRACAO COMPLETA DO CENARIO
SELECT *
FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60
WHERE `cod_centro` = '4128'
ORDER BY `cod_material`;

### 19.1 Alternativa para volume grande _(opcional)_

Descomente para gravar o resultado numa tabela própria e exportar de lá sem limite de tela.

In [ ]:
-- 19.1 GRAVAR EXTRACAO EM TABELA (opcional)
-- CREATE OR REPLACE TABLE dev_procurement.corp_curated.extracao_mm60_4128 AS
-- SELECT * FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128';
--
-- SELECT COUNT(*) FROM dev_procurement.corp_curated.extracao_mm60_4128;
SELECT 'Descomente as linhas acima se precisar gravar a extracao em tabela' AS instrucao;

## 20. Resumo do cenário

Bloco final. **Copie esta saída** e envie ao agente junto com o notebook.

In [ ]:
-- 20. RESUMO DO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128'),
g AS (
  SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'cod_material + cod_centro + tp_avaliacao' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `tp_avaliacao` FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'cod_material' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material` FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60 WHERE `cod_centro` = '4128')
)
SELECT 'CENARIO' AS bloco, 'transacao' AS item, 'MM60' AS valor
UNION ALL SELECT 'CENARIO', 'tabela', 'dev_procurement.corp_curated.tbl_ds_mdm_mm60'
UNION ALL SELECT 'CENARIO', 'filtro', 'cod_centro = 4128'
UNION ALL SELECT 'CENARIO', 'linhas no cenario', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'CENARIO', 'colunas', '17'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Baixar a **seção 19** em CSV — é a base do centro 4128 no Datalake.
2. Extrair a mesma transação no SAP com o filtro `centro = 4128`, **todas as abas**.
3. Anotar a data e hora das duas extrações.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Antes de comparar

- [ ] Zeros à esquerda normalizados nos dois lados (seção 12)
- [ ] Formato de data normalizado (seção 11)
- [ ] Totais numéricos conferidos (seção 10)
- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP (seção 6)
